# Data Selection

Loads the run-wise datacheck table produced by `1_create_total_dcheck.ipynb`,
applies configurable quality cuts (date range, dust, NSB, zenith distance, …),
matches subruns to a source wobble ring, and exports a clean run list.

**Pipeline:**
1. Load run-wise CSV + flat subrun parquet from notebook 1
2. (Optional) filter by date range
3. (Optional) filter by weather / dust / data-quality cuts
4. Match pointing to source wobble ring
5. Inspect & export selected run list

## Imports

In [ ]:
import os, glob, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.colors import LogNorm
import copy

import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation
import healpy as hp
from scipy.optimize import curve_fit
import utils

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore")

observing_location = EarthLocation.of_site("Roque de los Muchachos")

## Paths and configuration

All tunable parameters are gathered here.

In [ ]:
# ── Directory layout (must match notebook 1) ────────────────────────────
ROOT       = Path(os.getcwd())
ROOT_DATA  = ROOT / "data"
DCHECK_DIR = ROOT_DATA / "datachecks"
OUT_DIR    = ROOT_DATA / "selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Input files from notebook 1 ──────────────────────────────────────────
FNAME_DCHECK_FLAT_SRUNWISE = DCHECK_DIR / "datachecks_flat.parquet"         # subrun-level
FNAME_DCHECK_FLAT_RUNWISE = DCHECK_DIR / "datachecks_flat_runwise.parquet" # run-level

# ── Output files ─────────────────────────────────────────────────────────
FNAME_SELECTED_RUNS = OUT_DIR / "selected_runs.csv"

# ── Source / wobble ──────────────────────────────────────────────────────
SOURCE_NAME       = "Crab"
WOBBLE_DISTANCE   = 0.4 * u.deg
POINTING_UNCERT   = 0.1 * u.deg

# ── Date filter  (set to None to disable) ────────────────────────────────
DATE_MIN = datetime.fromisoformat("2025-08-17")   # e.g. datetime.fromisoformat("2024-01-01")
DATE_MAX = None   # e.g. datetime.fromisoformat("2025-06-01")

# ── Dust filter  (tng_dust in µg/m³; set to None to disable) ────────────
DUST_MAX = None

# ── Data-quality cuts (set each to None to disable individually) ─────────
ZD_MAX              = None   # max mean zenith distance [deg]
NSB_MAX             = None   # max diffuse_nsb_std  (arbitrary units; None = no cut)
ELAPSED_TIME_MIN    = 120.0  # min run duration [s]; removes very short runs
ELAPSED_TIME_MAX    = 2500.0  
DRDI_FIT_ERROR      = False  # True, drop runs where the DRDI fit failed

print("Configuration loaded.")
print(f"  Source          : {SOURCE_NAME}")
print(f"  Wobble distance : {WOBBLE_DISTANCE}")
print(f"  Date filter     : {DATE_MIN} -> {DATE_MAX}")
print(f"  Dust max        : {DUST_MAX} ug/m3")
print(f"  ZD max          : {ZD_MAX} deg")
print(f"  NSB max         : {NSB_MAX}")
print(f"  Elapsed time    : {ELAPSED_TIME_MIN/60:.1f} - {ELAPSED_TIME_MAX/60:.1f} min")


## Step 1 - Load run-wise table

In [ ]:
df = pd.read_parquet(FNAME_DCHECK_FLAT_RUNWISE)
df["time"] = pd.to_datetime(df["time"])

print(f"Loaded {len(df):} runs from {FNAME_DCHECK_FLAT_RUNWISE.name}")
print(f"Date range: {df['time'].min().date()} -> {df['time'].max().date()}")
print(f"\nColumns available: {list(df.columns)}\n")
df.head(3)

## Step 2 - Dataset overview

Quick look at the key quality variables before applying any cut.

In [ ]:
plot_cfg = [
    ("ZD_corrected_cosmics_rate_at_422_pe",  "ZD-corr. cosmics rate [Hz]", 40),
    ("diffuse_nsb_std",                      "Diffuse NSB std",            40),
    ("zd",                                   "Mean Zenith Distance [deg]", 40),
    ("tng_dust",                             "TNG Dust [$\\mu g/m^3$]",    40),
    ("humidity",                             "Humidity [%]",               40),
    ("corrected_elapsed_time",                             "Elapsed time [s]",           40),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, (col, label, bins) in zip(axes.ravel(), plot_cfg):
    if col in df.columns:
        data = df[col].dropna()
        ax.hist(data, bins=bins, color="lightgray", edgecolor="none", alpha=1)
        ax.set(xlabel=label, ylabel="Runs", yscale="log")
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f"{col!r}\nnot in table", ha="center", va="center", transform=ax.transAxes, color="gray")
        ax.axis("off")
plt.tight_layout()
plt.show()

## Step 3 - (Optional) Date filter

Set `DATE_MIN` / `DATE_MAX` in the configuration cell to restrict the sample to a specific observation period.  
Leave as `None` to keep the full dataset.

In [ ]:
df_sel = df.copy()
n_before = len(df_sel)

if DATE_MIN is not None:
    mask_date_min = df_sel["time"] >= pd.Timestamp(DATE_MIN)
    df_sel = df_sel[mask_date_min]
    print(f"  DATE_MIN cut ({DATE_MIN}): {n_before - len(df_sel):,} runs removed  "
          f"-> {len(df_sel):,} remaining")
    n_before = len(df_sel)
else:
    print("  DATE_MIN : no cut applied")

if DATE_MAX is not None:
    mask_date_max = df_sel["time"] <= pd.Timestamp(DATE_MAX)
    df_sel = df_sel[mask_date_max]
    print(f"  DATE_MAX cut ({DATE_MAX}): {n_before - len(df_sel):,} runs removed  "
          f"-> {len(df_sel):,} remaining")
else:
    print("  DATE_MAX : no cut applied")

print(f"After date filter: {len(df_sel):,} runs")


## Step 4 - (Optional) Dust & weather quality cuts

Filters based on the MAGIC Weather Station data merged in notebook 1.

- **`DUST_MAX`**: maximum TNG dust load in µg/m³.  
  The `tng_dust` column is the `RA` sensor from the WS (`Rain/Dust Activity` index).  
  A conservative clear-night threshold is **100 µg/m³**; relax to e.g. 500 during winter.  
  Set to `None` to skip this cut entirely.
- **`NSB_MAX`**: cap on the diffuse NSB standard deviation (moonlight / clouds proxy).
- **`ZD_MAX`**: maximum mean zenith distance in degrees (affects effective area & threshold).
- **`ELAPSED_TIME_MIN`**: removes stub runs (e.g. aborted runs < 2 min).
- **`DRDI_FIT_ERROR`**: drops runs where the intra-run rate stability fit failed.

In [ ]:
n_before = len(df_sel)
cut_log = []

# ── Dust ─────────────────────────────────────────────────────────────────
if DUST_MAX is not None and "tng_dust" in df_sel.columns:
    mask_dust = df_sel["tng_dust"].isna() | (df_sel["tng_dust"] <= DUST_MAX)
    n_cut = n_before - mask_dust.sum()
    df_sel = df_sel[mask_dust]
    cut_log.append(f"  Dust ≤ {DUST_MAX} ug/m3        : -{n_cut:>5,} runs  -> {len(df_sel):,} remaining")
    n_before = len(df_sel)
elif DUST_MAX is not None:
    cut_log.append("  Dust cut requested but tng_dust column not found - skipped")
else:
    cut_log.append("  Dust cut : disabled (DUST_MAX = None)")

# ── NSB ──────────────────────────────────────────────────────────────────
if NSB_MAX is not None and "diffuse_nsb_std" in df_sel.columns:
    mask_nsb = df_sel["diffuse_nsb_std"].isna() | (df_sel["diffuse_nsb_std"] <= NSB_MAX)
    n_cut = n_before - mask_nsb.sum()
    df_sel = df_sel[mask_nsb]
    cut_log.append(f"  NSB_std ≤ {NSB_MAX}              : -{n_cut:>5,} runs  -> {len(df_sel):,} remaining")
    n_before = len(df_sel)
else:
    cut_log.append("  NSB cut  : disabled (NSB_MAX = None)")

# ── Zenith distance ──────────────────────────────────────────────────────
if ZD_MAX is not None and "zd" in df_sel.columns:
    mask_zd = df_sel["zd"].isna() | (df_sel["zd"] <= ZD_MAX)
    n_cut = n_before - mask_zd.sum()
    df_sel = df_sel[mask_zd]
    cut_log.append(f"  ZD ≤ {ZD_MAX}°                  : -{n_cut:>5,} runs  -> {len(df_sel):,} remaining")
    n_before = len(df_sel)
else:
    cut_log.append("  ZD cut   : disabled (ZD_MAX = None)")

# ── Elapsed time ─────────────────────────────────────────────────────────
if "telapsed" in df_sel.columns and (ELAPSED_TIME_MIN is not None or ELAPSED_TIME_MAX is not None):
    # Apply min and max filters dynamically
    mask_tel = pd.Series(True, index=df_sel.index)
    if ELAPSED_TIME_MIN is not None:
        mask_tel &= df_sel["telapsed"] >= ELAPSED_TIME_MIN
    if ELAPSED_TIME_MAX is not None:
        mask_tel &= df_sel["telapsed"] <= ELAPSED_TIME_MAX

    n_cut = n_before - mask_tel.sum()
    df_sel = df_sel[mask_tel]
    
    # Construct a concise log string matching your style
    cond_str = f"{ELAPSED_TIME_MIN or 0} ≤ telapsed ≤ {ELAPSED_TIME_MAX or 'infinite'}"
    cut_log.append(f"  {cond_str:<22} : -{n_cut:>5,} runs  -> {len(df_sel):,} remaining")
    n_before = len(df_sel)
else:
    cut_log.append("  Elapsed time cut : disabled")

# ── DRDI fit error flag ───────────────────────────────────────────────────
if DRDI_FIT_ERROR and "run_drdi_fit_error_flag" in df_sel.columns:
    mask_drdi = ~df_sel["run_drdi_fit_error_flag"].astype(bool)
    n_cut = n_before - mask_drdi.sum()
    df_sel = df_sel[mask_drdi]
    cut_log.append(f"  DRDI fit error removed         : -{n_cut:>5,} runs  -> {len(df_sel):,} remaining")
else:
    cut_log.append("  DRDI fit error cut : disabled")

print("\n── Quality cut summary ──────────────────────────────────────────────")
for line in cut_log:
    print(line)
print("─────────────────────────────────────────────────────────────────────")
print(f"After quality cuts: {len(df_sel):,} runs  "
      f"({len(df_sel)/len(df)*100:.1f}% of full dataset)")


## Step 5 - AltAz -> RA/Dec conversion and wobble-ring matching

In [ ]:
# Wobble ring matching
source_coord = SkyCoord.from_name(SOURCE_NAME)
run_coords   = SkyCoord(ra=df_sel["ra"].to_numpy() * u.deg, dec=df_sel["dec"].to_numpy() * u.deg)
separation   = source_coord.separation(run_coords)

min_dist = WOBBLE_DISTANCE - POINTING_UNCERT
max_dist = WOBBLE_DISTANCE + POINTING_UNCERT
mask_wobble = (separation >= min_dist) & (separation <= max_dist)

df_sel["separation_deg"] = separation.deg
df_sel = df_sel[mask_wobble].copy()

print(f"Wobble ring [{min_dist:.2f}, {max_dist:.2f}] around {SOURCE_NAME}")
print(f"  Source: RA={source_coord.ra.deg:.3f} deg, Dec={source_coord.dec.deg:.3f} deg")
print(f"  Runs matching wobble ring: {len(df_sel):,}")
print(f"  Total elapsed time       : {df_sel["corrected_elapsed_time"].sum()/3600:.2f} h")

## Step 6 - Pointing diagnostics

In [ ]:
# ── AltAz polar map ──────────────────────────────────────────────────────
theta = np.radians(np.array(df_sel["az"].to_numpy()) % 360.0)
r     = df_sel["zd"].to_numpy()

counts, theta_edges, r_edges = np.histogram2d(
    theta, r, bins=[60, 30], range=[[0, 2 * np.pi], [0, 90]]
)
Thetas, Rs = np.meshgrid(theta_edges, r_edges)

# ── HEALPix sky map ──────────────────────────────────────────────────────
healpix_map = np.zeros(hp.nside2npix(32))
theta_hp = np.radians(90.0 - df_sel["dec"].values)
phi_hp   = np.radians(df_sel["ra"].values)
np.add.at(healpix_map, hp.ang2pix(32, theta_hp, phi_hp), 1)
healpix_map[healpix_map == 0] = np.nan

fig_polar, ax_polar = plt.subplots(figsize=(5, 5), subplot_kw={"projection": "polar"})
pc = ax_polar.pcolormesh(Thetas, Rs, counts.T, cmap="viridis", norm=LogNorm(vmin=0.5))
plt.colorbar(pc, orientation="horizontal", pad=0.1, label="Counts")
ax_polar.set(theta_zero_location="N", theta_direction=-1,
             rmax=90, title="Alt - Az", rticks=[90, 60, 30, 0],
             yticklabels=["90°", "60°", "30°", "0°"])
fig_polar.canvas.draw()
plt.close(fig_polar)

fig_hp = plt.figure(figsize=(6, 4), facecolor="white")
cmap_hp = copy.copy(plt.get_cmap("viridis"))
cmap_hp.set_under("white")
hp.mollview(healpix_map, coord=["C", "G"], title="Galactic Lon - Lat",
            unit="\nCounts", cmap=cmap_hp, norm="log", hold=True)
hp.graticule(dpar=30, dmer=45, coord="G", local=True)
fig_hp.canvas.draw()
plt.close(fig_hp)

fig_final, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(13, 6), facecolor="white",
    gridspec_kw={"width_ratios": [1, 1.5]}
)
ax1.imshow(fig_polar.canvas.buffer_rgba()); ax2.imshow(fig_hp.canvas.buffer_rgba())
ax1.axis("off"); ax2.axis("off")
pos2 = ax2.get_position()
ax2.set_position([pos2.x0, pos2.y0 + 0.05, pos2.width, pos2.height])
plt.suptitle(f"Selected runs pointing - {SOURCE_NAME} wobble ring", y=1.01)
plt.tight_layout()
plt.show()

# ── Wobble ring scatter ───────────────────────────────────────────────────
azimuths_ring = np.linspace(0, 360, 300) * u.deg
circle = source_coord.directional_offset_by(azimuths_ring, WOBBLE_DISTANCE)
circle_in  = source_coord.directional_offset_by(azimuths_ring, WOBBLE_DISTANCE - POINTING_UNCERT)
circle_out = source_coord.directional_offset_by(azimuths_ring, WOBBLE_DISTANCE + POINTING_UNCERT)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df_sel["ra"], df_sel["dec"], s=8, alpha=0.6,
           c=df_sel["zd"], cmap="plasma", label="Selected runs")
ax.plot(source_coord.ra.deg, source_coord.dec.deg, "*r", label=SOURCE_NAME)
ax.plot(circle.ra.deg, circle.dec.deg, "r--", lw=1.5, label=f"Wobble ring ({WOBBLE_DISTANCE.value:.2f} deg)")
ax.plot(circle_in.ra.deg, circle_in.dec.deg, "r:", lw=0.8)
ax.plot(circle_out.ra.deg, circle_out.dec.deg, "r:", lw=0.8,
        label=f"±{POINTING_UNCERT.value:.2f} deg tolerance")
sm = plt.cm.ScalarMappable(cmap="plasma",
     norm=plt.Normalize(df_sel["zd"].min(), df_sel["zd"].max()))
plt.colorbar(sm, ax=ax, label="Mean ZD [deg]")
ax.set(xlabel="RA [deg]", ylabel="Dec [deg]", title=f"{len(df_sel)} selected runs around {SOURCE_NAME}")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.3, ls=":")
plt.tight_layout()
plt.show()

## Step 7 - Timeline & dust inspection

Visualise the selected runs over time together with key quality variables.
This is particularly useful to check whether cuts on dust or date are sensible.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=True)

ts = df_sel["time"]

# Panel 1 – ZD-corrected cosmics rate
axes[0].scatter(ts, df_sel["ZD_corrected_cosmics_rate_at_422_pe"], s=6, alpha=0.5, color="royalblue")
axes[0].set(ylabel="ZD-corr. rate\n[Hz]")
axes[0].grid(alpha=0.3)

# Panel 2 – Dust
if "tng_dust" in df_sel.columns:
    axes[1].scatter(ts, df_sel["tng_dust"], s=6, alpha=0.5, color="saddlebrown")
    if DUST_MAX is not None:
        axes[1].axhline(DUST_MAX, color="red", lw=1, ls="--",
                        label=f"DUST_MAX = {DUST_MAX}")
        axes[1].legend(fontsize=8)
    axes[1].set(ylabel="TNG Dust\n[$\\mu g/m^3$]", yscale="log")
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "tng_dust column not available",
                 ha="center", va="center", transform=axes[1].transAxes, color="gray")

# Panel 3 – NSB
if "light_yield" in df_sel.columns:
    axes[2].scatter(ts, df_sel["light_yield"], s=6, alpha=0.5, color="darkorange")
    axes[2].set(ylabel="Light Yield", ylim=(0.5, 1.2))
    axes[2].grid(alpha=0.3)

# Panel 4 – Elapsed time per run
axes[3].scatter(ts, df_sel["corrected_elapsed_time"] / 60, s=6, alpha=0.5, color="seagreen")
if ELAPSED_TIME_MIN is not None:
    axes[3].axhline(ELAPSED_TIME_MIN / 60, color="red", lw=1, ls="--",
                    label=f"min = {ELAPSED_TIME_MIN/60:.1f} min")
    axes[3].legend(fontsize=8)
axes[3].set(ylabel="Elapsed time\n[min]", xlabel="Date")
axes[3].grid(alpha=0.3)
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%b"))
axes[3].xaxis.set_major_locator(mdates.MonthLocator(interval=2))

fig.suptitle(f"Selected runs - {SOURCE_NAME}")
plt.gcf().autofmt_xdate(rotation=0, ha="center")
plt.tight_layout()
plt.show()

converted_dates = mdates.date2num(df_sel["time"])

fig, ax = plt.subplots(figsize=(5, 3.5))
sc = ax.scatter(df_sel["tng_dust"], df_sel["light_yield"], marker=".", s=df_sel["corrected_elapsed_time"]**0.7, c=converted_dates)
cbar = fig.colorbar(sc, ax=ax)
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%Y-%b-%d"))
ax.grid(alpha=0.3)
ax.set(xlabel="TNG dust [$\\mu g /m^3$]", ylabel="Intensity light yield", ylim=(0.6, 1.1), xlim=(0.5e-2, 30), xscale="log")
plt.show()

## Step 8 - Run list summary

In [ ]:
# Sort by date
df_sel = df_sel.sort_values("time").reset_index(drop=True)

run_list = df_sel[["obs_id", "time", "n_subruns", "corrected_elapsed_time",
                   "zd", "separation_deg", "ZD_corrected_cosmics_rate_at_422_pe",
                   "diffuse_nsb_std", "tng_dust"]]

print("── Selected run list ────────────────────────────────────────────────")
for _, row in run_list.iterrows():
    date_str = str(row["time"]).split("T")[0].split(" ")[0]
    print(f"  {date_str}  run {int(row["obs_id"]):6d}  "
          f"zd={row['zd']:5.1f} deg  t={row['corrected_elapsed_time']/60:5.1f} "
          f"min dust={row['tng_dust']:02.1f} ug/m3")

print("─────────────────────────────────────────────────────────────────────")
print(f"Total: {len(df_sel):,} runs  |  {df_sel['corrected_elapsed_time'].sum()/3600:.2f} h  |  "
      f"{df_sel['n_subruns'].sum():,} subruns")

print(f"\nUnique run numbers:")
print(sorted(df_sel["obs_id"].astype(int).tolist()))


## Step 9 - Export selected run list

In [ ]:
df_sel.to_csv(FNAME_SELECTED_RUNS, index=False)
print(f"Saved {len(df_sel):,} selected runs → {FNAME_SELECTED_RUNS}")

# Also print a compact summary
print("\n── Selection summary ────────────────────────────────────────────────")
print(f"  Source              : {SOURCE_NAME}")
print(f"  Wobble distance     : {WOBBLE_DISTANCE}")
print(f"  Date range applied  : {DATE_MIN}  →  {DATE_MAX}")
print(f"  Dust cut (DUST_MAX) : {DUST_MAX} ug/m3")
print(f"  NSB cut  (NSB_MAX)  : {NSB_MAX}")
print(f"  ZD cut   (ZD_MAX)   : {ZD_MAX} deg")
print(f"  Min elapsed time    : {ELAPSED_TIME_MIN} s")
print(f"  ─────────────────────────────────────")
print(f"  Input runs          : {len(df):,}")
print(f"  Selected runs       : {len(df_sel):,}  ({len(df_sel)/len(df)*100:.1f}%)")
print(f"  Total elapsed time  : {df_sel['corrected_elapsed_time'].sum()/3600:.2f} h")
print(f"  Total subruns       : {df_sel['n_subruns'].sum():,}")
print("─────────────────────────────────────────────────────────────────────")

display(df_sel.describe().T[["mean", "std", "min", "max"]])

## Step 10 - Run-wise charge yield summary

For each selected run, plot the subrun-wise charge yield over time together with a linear fit.
A green dashed line indicates a successful fit; red means the fit failed and the per-run median is used instead.
The TNG dust value is annotated in the top-left corner (green < 1.5, orange 1.5–3.0, red > 3.0 µg m⁻³).

In [ ]:
THR_FIT = 1e-3

for obs_id, row in df_sel.iterrows():
    # ── Locate the subrun-level parquet for this run ──────────────────────
    if not FNAME_DCHECK_FLAT_SRUNWISE.exists():
        print(f"Flat parquet not found at {FNAME_DCHECK_FLAT_SRUNWISE}, skipping run-wise plots.")
        break

    df_flat = pd.read_parquet(FNAME_DCHECK_FLAT_SRUNWISE)
    run_id  = int(row["obs_id"])

    tab = df_flat[df_flat["obs_id"] == run_id].copy()
    if tab.empty:
        print(f"  Run {run_id}: no subrun data found in flat parquet - skipping.")
        continue

    # ── Build arrays ──────────────────────────────────────────────────────
    # FIXED: Ensure tab["time"] is treated as actual datetimes by Pandas
    pd_time = pd.to_datetime(tab["time"])
    
    # Unix seconds (or scaled seconds depending on your 1e9 preference)
    tstamp       = np.array([t.timestamp() for t in pd_time]) / 1e9          
    dtime        = pd_time.to_numpy() # High-precision datetime64 array for plotting
    charge_yield = tab["light_yield"].to_numpy(dtype=float)
    dust_value   = float(row["tng_dust"]) if "tng_dust" in row and not pd.isna(row["tng_dust"]) else np.nan

    # ── Masking for the fit (drop NaNs and > 2-sigma outliers) ────────────
    mask_nan  = np.isnan(tstamp) | np.isnan(charge_yield)
    mean_s, std_s = np.nanmean(charge_yield), np.nanstd(charge_yield)
    mask_hi   = charge_yield > 2.0
    mask_out  = (charge_yield > mean_s + 2 * std_s) | (charge_yield < mean_s - 2 * std_s)
    mask_fit  = ~(mask_nan | mask_hi | mask_out)

    x_fit = tstamp[mask_fit]
    y_fit = charge_yield[mask_fit]

    fit_succesful = False
    fit_run_x0 = fit_run_x1 = np.nan
    fit_run_chi2 = fit_run_ndf = np.nan

    if len(x_fit) >= 2:
        try:
            params, pcov, info, _, _ = curve_fit(
                f= utils.straight_line, xdata=x_fit, ydata=y_fit,
                p0=[np.nanmean(y_fit), 0], full_output=True,
            )
            fit_run_x0, fit_run_x1 = params
            fit_run_chi2 = float(np.sum(info["fvec"] ** 2))
            fit_run_ndf  = len(x_fit)
            fit_succesful = (fit_run_chi2 / fit_run_ndf < THR_FIT) and (not np.isnan(fit_run_x0))
        except Exception:
            pass

    y_median = np.nanmedian(charge_yield)

    # ── Plot ──────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 3))

    ax.errorbar(
        dtime[mask_fit], charge_yield[mask_fit],
        ls="", marker="",
    )
    ylims = ax.get_ylim()

    # full trace in black
    ax.errorbar(dtime, charge_yield, ls="-", marker="", color="k", zorder=-10, label="Data",)
    ax.set_ylim(*ylims)

    # ── Generate fit line x-points ────────────────────────────────────────
    if len(tstamp) > 0 and not np.isnan(tstamp).all():
        xarray_ts = np.linspace(np.nanmin(tstamp), np.nanmax(tstamp), 200)
        unix_seconds = xarray_ts * 1e9
        xarray = np.array([datetime.fromtimestamp(ts) for ts in unix_seconds])
    else:
        xarray = np.array([])
        xarray_ts = np.array([])

    if fit_succesful:
        fit_label = (f"L.fit: OK\n$\\chi^2$/ndf={fit_run_chi2/fit_run_ndf:.0e}")
        ax.plot(xarray, utils.straight_line(xarray_ts, fit_run_x0, fit_run_x1), color="g", ls="--", label=fit_label)
    else:
        fit_label = (f"L.fit: not OK\n$\\chi^2$/ndf={fit_run_chi2/fit_run_ndf:.0e}"
            if not np.isnan(fit_run_chi2) else "L.fit: not OK"
        )
        if not np.isnan(fit_run_x0) and not np.isnan(fit_run_x1):
            ax.plot(xarray, utils.straight_line(xarray_ts, fit_run_x0, fit_run_x1), color="r", ls="--", label=fit_label)
        else:
            ax.plot([], [], color="r", ls="--", label=fit_label)
            
        ax.axhline(y_median, color="darkorange", ls=":", label=f"Using median\n{y_median:.3f}")

    ax.legend(frameon=False, loc=(1.03, 0))
    if not np.isnan(dust_value):
        cdust = "r" if dust_value > 3.0 else "g" if dust_value < 1.5 else "darkorange"
        ax.text(0.03, 1.05, f"TNG dust: {dust_value:.2f}" + "$\\mu$g m$^{-3}$",
            ha="left", va="center", transform=ax.transAxes, color=cdust,)

    ax.set(xlabel=f"Time UTC ({pd.Timestamp(dtime[0]).date()})", ylabel="Light Yield", title=f"Run {run_id}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.tight_layout()
    plt.show()


### B3 summary

In [ ]:
THR_FIT = 1e-3

for obs_id, row in df_sel.iterrows():
    # ── Locate the subrun-level parquet for this run ──────────────────────
    if not FNAME_DCHECK_FLAT_SRUNWISE.exists():
        print(f"Flat parquet not found at {FNAME_DCHECK_FLAT_SRUNWISE}, skipping run-wise plots.")
        break

    df_flat = pd.read_parquet(FNAME_DCHECK_FLAT_SRUNWISE)
    run_id  = int(row["obs_id"])

    tab = df_flat[df_flat["obs_id"] == run_id].copy()
    if tab.empty:
        print(f"  Run {run_id}: no subrun data found in flat parquet - skipping.")
        continue

    # ── Build arrays ──────────────────────────────────────────────────────
    # FIXED: Ensure tab["time"] is treated as actual datetimes by Pandas
    pd_time = pd.to_datetime(tab["time"])
    
    # Unix seconds (or scaled seconds depending on your 1e9 preference)
    tstamp       = np.array([t.timestamp() for t in pd_time]) / 1e9          
    dtime        = pd_time.to_numpy() # High-precision datetime64 array for plotting
    charge_yield = tab["ly_b3"].to_numpy(dtype=float)
    dust_value   = float(row["tng_dust"]) if "tng_dust" in row and not pd.isna(row["tng_dust"]) else np.nan

    # ── Masking for the fit (drop NaNs and > 2-sigma outliers) ────────────
    mask_nan  = np.isnan(tstamp) | np.isnan(charge_yield)
    mean_s, std_s = np.nanmean(charge_yield), np.nanstd(charge_yield)
    mask_hi   = charge_yield > 2.0
    mask_out  = (charge_yield > mean_s + 2 * std_s) | (charge_yield < mean_s - 2 * std_s)
    mask_fit  = ~(mask_nan | mask_hi | mask_out)

    x_fit = tstamp[mask_fit]
    y_fit = charge_yield[mask_fit]

    fit_succesful = False
    fit_run_x0 = fit_run_x1 = np.nan
    fit_run_chi2 = fit_run_ndf = np.nan

    if len(x_fit) >= 2:
        try:
            params, pcov, info, _, _ = curve_fit(
                f= utils.straight_line, xdata=x_fit, ydata=y_fit,
                p0=[np.nanmean(y_fit), 0], full_output=True,
            )
            fit_run_x0, fit_run_x1 = params
            fit_run_chi2 = float(np.sum(info["fvec"] ** 2))
            fit_run_ndf  = len(x_fit)
            fit_succesful = (fit_run_chi2 / fit_run_ndf < THR_FIT) and (not np.isnan(fit_run_x0))
        except Exception:
            pass

    y_median = np.nanmedian(charge_yield)

    # ── Plot ──────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 3))

    ax.errorbar(
        dtime[mask_fit], charge_yield[mask_fit],
        ls="", marker="",
    )
    ylims = ax.get_ylim()

    # full trace in black
    ax.errorbar(dtime, charge_yield, ls="-", marker="", color="k", zorder=-10, label="Data",)
    ax.set_ylim(*ylims)

    # ── Generate fit line x-points ────────────────────────────────────────
    if len(tstamp) > 0 and not np.isnan(tstamp).all():
        xarray_ts = np.linspace(np.nanmin(tstamp), np.nanmax(tstamp), 200)
        unix_seconds = xarray_ts * 1e9
        xarray = np.array([datetime.fromtimestamp(ts) for ts in unix_seconds])
    else:
        xarray = np.array([])
        xarray_ts = np.array([])

    if fit_succesful:
        fit_label = (f"L.fit: OK\n$\\chi^2$/ndf={fit_run_chi2/fit_run_ndf:.0e}")
        ax.plot(xarray, utils.straight_line(xarray_ts, fit_run_x0, fit_run_x1), color="g", ls="--", label=fit_label)
    else:
        fit_label = (f"L.fit: not OK\n$\\chi^2$/ndf={fit_run_chi2/fit_run_ndf:.0e}"
            if not np.isnan(fit_run_chi2) else "L.fit: not OK"
        )
        if not np.isnan(fit_run_x0) and not np.isnan(fit_run_x1):
            ax.plot(xarray, utils.straight_line(xarray_ts, fit_run_x0, fit_run_x1), color="r", ls="--", label=fit_label)
        else:
            ax.plot([], [], color="r", ls="--", label=fit_label)
            
        ax.axhline(y_median, color="darkorange", ls=":", label=f"Using median\n{y_median:.3f}")

    ax.legend(frameon=False, loc=(1.03, 0))
    if not np.isnan(dust_value):
        cdust = "r" if dust_value > 3.0 else "g" if dust_value < 1.5 else "darkorange"
        ax.text(0.03, 1.05, f"TNG dust: {dust_value:.2f}" + "$\\mu$g m$^{-3}$",
            ha="left", va="center", transform=ax.transAxes, color=cdust,)

    ax.set(xlabel=f"Time UTC ({pd.Timestamp(dtime[0]).date()})", ylabel="Light Yield", title=f"Run {run_id}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.tight_layout()
    plt.show()
